In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.nn import parameter
%matplotlib inline

words = open('names.txt', 'r').read().splitlines()

chars = sorted(list(set(''.join(words))))
stringToIntegerMap = {s:i+1 for i,s in enumerate(chars)}
stringToIntegerMap['.'] = 0
integerToStrigMapping = {s:i for i,s in stringToIntegerMap.items()}
integerToStrigMapping

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [32]:
#create the training dataset

block_size = 3 # context length: how many characters do we take to predict the next one ?
X, Y = [], []

# emma
# ... e
# ..e m
# .em a
# emm a

for word in words:
    print(word)
    context = [0] * block_size
    for ch in word + ".":
        xIndex = stringToIntegerMap[ch]
        X.append(context)
        Y.append(xIndex)
        print("".join(integerToStrigMapping[i] for i in context), '--->', integerToStrigMapping[xIndex])
        context = context[1:] + [xIndex]

X = torch.tensor(X)
Y = torch.tensor(Y)


emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .
charlotte
... ---> c
..c ---> h
.ch ---> a
cha ---> r
har ---> l
arl ---> o
rlo ---> t
lot ---> t
ott ---> e
tte ---> .
mia
... ---> m
..m ---> i
.mi ---> a
mia ---> .
amelia
... ---> a
..a ---> m
.am ---> e
ame ---> l
mel ---> i
eli ---> a
lia ---> .
harper
... ---> h
..h ---> a
.ha ---> r
har ---> p
arp ---> e
rpe ---> r
per ---> .
evelyn
... ---> e
..e ---> v
.ev ---> e
eve ---> l
vel ---> y
ely ---> n
lyn ---> .
abigail
... ---> a
..a ---> b
.ab ---> i
abi ---> g
big ---> a
iga ---> i
gai ---> l
ail ---> .
emily
... ---> e
..e ---> m
.em ---> i
emi ---> l
mil ---> y
ily ---> .
elizabeth
... ---> e
..e ---

In [6]:
X.shape

torch.Size([32, 3])

In [7]:
Y.shape

torch.Size([32])

In [10]:
C = torch.randn((27,2))
C

tensor([[-1.1399, -0.4122],
        [-0.8525,  0.8862],
        [ 0.2326,  0.6558],
        [-1.3219, -2.1408],
        [ 0.3062, -0.5875],
        [ 0.7044, -0.4001],
        [-0.4302,  0.4370],
        [ 1.4138,  1.7369],
        [-1.8230,  0.6996],
        [-0.2257, -0.7715],
        [-1.5684, -0.9115],
        [ 0.1916,  1.2326],
        [ 1.1405,  1.2217],
        [ 1.2735,  2.5075],
        [-0.4400,  0.2691],
        [ 0.1272,  0.0864],
        [ 0.0081,  1.4730],
        [-1.2883,  1.1821],
        [ 1.7682,  0.1579],
        [ 0.7801, -1.8306],
        [-1.3097,  0.5286],
        [ 2.6683, -0.6854],
        [ 1.1051, -0.4266],
        [ 0.6511,  0.0489],
        [ 1.4773, -2.9032],
        [-0.8826,  0.0920],
        [ 0.7715,  0.9353]])

In [11]:
C[X].shape

torch.Size([32, 3, 2])

In [13]:
emb = C[X]

In [14]:
W1 = torch.randn((6, 100))
b1 = torch.randn((100))

In [15]:
emb @ W1 + b1

RuntimeError: mat1 and mat2 shapes cannot be multiplied (96x2 and 6x100)

In [18]:
h = torch.tanh(emb.view(32,6) @ W1 + b1)

In [20]:
W2 = torch.randn((100, 27))
b2 = torch.randn((27))

In [21]:
logits = h @ W2 + b2

In [22]:
counts = logits.exp()
prob = counts/ counts.sum(1, keepdim=True)
prob.shape

torch.Size([32, 27])

In [23]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [24]:
loss = -prob[torch.arange(32),Y].log().mean()

In [34]:
 # Summary
g = torch.Generator()
g.manual_seed(0).manual_seed(2147483647)

C = torch.randn((27,2), generator= g)

W1 = torch.randn((6, 100), generator= g)
b1 = torch.randn((100), generator= g)

W2 = torch.randn((100, 27), generator= g)
b2 = torch.randn((27), generator= g)

parameters = [C, W1, b1, W2, b2]

sum (p.nelement() for p in parameters)

3481

In [35]:
emb = C[X]
h = torch.tanh(emb.view(emb.shape[0],6) @ W1 + b1)
logits = h @ W2 + b2

counts = logits.exp()
prob = counts/ counts.sum(1, keepdim=True)
loss = -prob[torch.arange(32),Y].log().mean()
loss

IndexError: shape mismatch: indexing tensors could not be broadcast together with shapes [32], [228146]

In [29]:
F.cross_entropy(logits, Y)

tensor(17.7697)

In [39]:
for p in parameters:
    p.requires_grad = True

In [41]:
for _ in range(100):
    #forward pass
    emb = C[X]
    h = torch.tanh(emb.view(emb.shape[0],6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y)
    print(loss.item())

    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    #update
    for p in parameters:
        p.data += -0.1 * p.grad

3.575547456741333
3.5621628761291504
3.549084424972534
3.5363051891326904
3.5238142013549805
3.511603593826294
3.4996650218963623
3.4879894256591797
3.4765703678131104
3.4653990268707275
3.4544684886932373
3.4437711238861084
3.433300495147705
3.4230501651763916
3.4130125045776367
3.403181791305542
3.39355206489563
3.3841168880462646
3.3748719692230225
3.365809679031372
3.356926918029785
3.348217010498047
3.3396759033203125
3.331298589706421
3.3230810165405273
3.315018653869629
3.3071060180664062
3.2993409633636475
3.2917182445526123
3.2842342853546143
3.276885509490967
3.2696683406829834
3.2625787258148193
3.2556138038635254
3.2487707138061523
3.2420456409454346
3.2354350090026855
3.2289369106292725
3.2225475311279297
3.216264486312866
3.2100841999053955
3.204005002975464
3.198023796081543
3.1921377182006836
3.1863441467285156
3.1806414127349854
3.1750266551971436
3.16949725151062
3.1640517711639404
3.158687114715576
3.1534011363983154
3.148193597793579
3.1430604457855225
3.13800048828

In [38]:

block_size = 3 # context length: how many characters do we take to predict the next one ?
X, Y = [], []

for word in words:
    print(word)
    context = [0] * block_size
    for ch in word + ".":
        xIndex = stringToIntegerMap[ch]
        X.append(context)
        Y.append(xIndex)
        #print("".join(integerToStrigMapping[i] for i in context), '--->', integerToStrigMapping[xIndex])
        context = context[1:] + [xIndex]

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
olivia
ava
isabella
sophia
charlotte
mia
amelia
harper
evelyn
abigail
emily
elizabeth
mila
ella
avery
sofia
camila
aria
scarlett
victoria
madison
luna
grace
chloe
penelope
layla
riley
zoey
nora
lily
eleanor
hannah
lillian
addison
aubrey
ellie
stella
natalie
zoe
leah
hazel
violet
aurora
savannah
audrey
brooklyn
bella
claire
skylar
lucy
paisley
everly
anna
caroline
nova
genesis
emilia
kennedy
samantha
maya
willow
kinsley
naomi
aaliyah
elena
sarah
ariana
allison
gabriella
alice
madelyn
cora
ruby
eva
serenity
autumn
adeline
hailey
gianna
valentina
isla
eliana
quinn
nevaeh
ivy
sadie
piper
lydia
alexa
josephine
emery
julia
delilah
arianna
vivian
kaylee
sophie
brielle
madeline
peyton
rylee
clara
hadley
melanie
mackenzie
reagan
adalynn
liliana
aubree
jade
katherine
isabelle
natalia
raelynn
maria
athena
ximena
arya
leilani
taylor
faith
rose
kylie
alexandra
mary
margaret
lyla
ashley
amaya
eliza
brianna
bailey
andrea
khloe
jasmine
melody
iris
isabel
norah
annabelle
valeria
emerson
adalyn
ryl

In [42]:
g = torch.Generator().manual_seed(2147483647)

for _ in range(20):

    out = []
    context = [0] * block_size  # initialize with all ...
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break

    print(''.join(integerToStrigMapping[i] for i in out))


dee.
mazealserkioa.
ktyh.
mmmzzmitna.
nrllwde.
ka.
aa.
snaivaeelathr.
iotai.
iolieqlkauvn.
jted.
aka.
eyde.
sadly.
akavgynfrntls.
mhuonden.
tahlvsu.
dsdr.
dan.
gahayilaigana.
